# Amazon US Customer Reviews: Data Exploration

This notebook explores the data in the [Amazon US Customer Reviews](https://www.kaggle.com/datasets/cynthiarempel/amazon-us-customer-reviews-dataset) dataset sourced from Kaggle. The dataset is approximately 50.68 GB and contains over 109 million rows across dozens of product categories. We examine the data structure, distributions, missing values, duplicates, and key relationships between review attributes.

## Setup and Imports

In [1]:
from pyspark.sql import SparkSession, SQLContext
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time
from pyspark.sql import SQLContext
from pyspark.sql.functions import col, length, round, expr, sum as spark_sum
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler

Matplotlib created a temporary cache directory at /scratch/ajaganathan/job_49189747/matplotlib-j5pigxn2 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


## Spark Session Configuration

Our SDSC Expanse allocation provides 16 total cores and 128 GB total memory. The dataset is 50.68 GB across 37 Parquet files. 

In [2]:
import subprocess, os
USER = os.getenv("USER")
subprocess.run(["mkdir", "-p", f"/expanse/lustre/projects/uci157/{USER}/spark_tmp"])

CompletedProcess(args=['mkdir', '-p', '/expanse/lustre/projects/uci157/ajaganathan/spark_tmp'], returncode=0)

In [3]:
# Template: Adjust TOTAL_MEMORY and TOTAL_CORES to match your allocation
TOTAL_MEMORY = 128  # GB - from your Jupyter request
TOTAL_CORES = 16     # from your Jupyter request
DRIVER_MEMORY = 4   # GB - fixed, small (driver doesn't process data)

num_executors = TOTAL_CORES - 1
executor_memory = (TOTAL_MEMORY - DRIVER_MEMORY) // num_executors

spark = SparkSession.builder \
    .config("spark.driver.memory", f"{DRIVER_MEMORY}g") \
    .config("spark.executor.memory", f"{executor_memory}g") \
    .config("spark.executor.instances", num_executors) \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.local.dir", f"/expanse/lustre/projects/uci157/{USER}/spark_tmp") \
    .getOrCreate()

spark

### Executor Verification

In [5]:
# Get the active Spark Context, URL, and SQL Context
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"
sqlContext = SQLContext(spark.sparkContext)

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
spd = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
spd['maxMemory_GB'] = (spd['maxMemory'] / (1024**3)).round(2)
spd

/usr/local/spark/python/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,16,2388236697,0,True,2.22


### Current Spark Configuration

In [6]:
print("Executor Instances:", sc.getConf().get("spark.executor.instances"))
print("Executor Memory:", sc.getConf().get("spark.executor.memory"))
print("Driver Memory:", sc.getConf().get("spark.driver.memory"))
print("Executor Cores:", sc.getConf().get("spark.executor.cores"))
print("Total Cores Available:", sc._jsc.sc().defaultParallelism())

Executor Instances: 15
Executor Memory: 8g
Driver Memory: 4g
Executor Cores: None
Total Cores Available: 16


## Data Loading

In [7]:
# Load the data into Spark dataframe
parquet_path = r"/expanse/lustre/projects/uci157/hkwon2/shared/amazon_reviews.parquet"
df = spark.read.parquet(parquet_path)

## 1. Complete Preprocessing using Spark (8 points)

### 1) Clean below invalid data (Anuradha)
1. Product category has invalid data like dates and reviews.
2. Star ratings has Null and dates
3. Null records in "review_id", "review_body", "review_headline", "review_date" and "product_category"
4. Duplicate review ids. Reviews have to be unique for analysis

In [8]:
def filter_invalid_categories(df):
    valid_categories = [
        "Wireless", "PC", "Mobile_Apps", "Digital_Ebook_Pur...", "Video DVD",
        "Apparel", "Music", "Health & Personal...", "Beauty", "Digital_Video_Dow...",
        "Toys", "Sports", "Shoes", "Books", "Automotive", "Electronics",
        "Office Products", "Pet Products", "Grocery", "Outdoors", "Camera",
        "Video Games", "Digital_Music_Pur...", "Baby", "Tools", "Watches",
        "Musical Instruments", "Furniture", "Video", "Software", "Gift Card",
        "Digital_Video_Games", "Mobile_Electronics", "Digital_Software",
        "Major Appliances", "Personal_Care_App...", "Home Entertainment",
        "Home Improvement", "Home", "Kitchen", "Lawn and Garden", "Luggage"
    ]
    return df.filter(col("product_category").isin(valid_categories))


def filter_invalid_star_ratings(df):
    valid_ratings = [1, 2, 3, 4, 5]
    return df.withColumn("star_rating", col("star_rating").cast("int")) \
             .withColumn("helpful_votes", col("helpful_votes").cast("int")) \
             .withColumn("total_votes", col("total_votes").cast("int")) \
             .filter(col("star_rating").isin(valid_ratings))


def drop_null_records(df):
    required_columns = ["review_id", "review_body", "review_headline", "review_date", "product_category"]
    return df.dropna(subset=required_columns)


def drop_duplicate_reviews(df):
    return df.dropDuplicates(["review_id"])

def impute_missing_values(df):
    imputer = Imputer(
        strategy="median",
        inputCols=["star_rating", "helpful_votes", "total_votes"],
        outputCols=["star_rating", "helpful_votes", "total_votes"]
    )
    return imputer.fit(df).transform(df)

In [ ]:
def cleansing1(df):
    df = filter_invalid_categories(df)
    df = filter_invalid_star_ratings(df)
    df = drop_null_records(df)
    df = drop_duplicate_reviews(df)
    df = impute_missing_values(df)
    return df

### 2) Numerical Encoding (Matthew)

- Cast star_rating from String to Integer for numerical processing. (Just in case we calculate the mean or so)
- Apply StringIndexer to encode categorical variables (verified_purchase, vine) into binary (0/1).

In [ ]:
def cleansing2(df):
    df = df.withColumn("star_rating", col("star_rating").cast("int"))
    df = df.withColumn("review_headline_length", length(col("review_headline")))
    
    indexer_vp = StringIndexer(inputCol="verified_purchase", outputCol="verified_purchase_idx", handleInvalid="keep")
    df = indexer_vp.fit(df).transform(df)
    
    category_indexer = StringIndexer(
        inputCol="product_category", outputCol="category_idx", handleInvalid="keep"
    )
    df = category_indexer.fit(df).transform(df)

    return df

### 3) Text Analytics & Scaling (Hana)

- Compute review_length using the length() function on the review_body column.
- Implement StandardScaler or MinMaxScaler to normalize numerical features for better model convergence.
- (Optional) Prepare a Tokenizer for potential text-based model expansion in Milestone 4.

In [ ]:
from pyspark.ml.feature import MinMaxScaler, Tokenizer

def cleansing3(df):
    df = df.withColumn("review_length", length(col("review_body")))
    # numeric_cols = ["star_rating", "helpful_votes", "total_votes", "review_length"]
    # assembler = VectorAssembler(inputCols=numeric_cols, outputCol="numeric_features", handleInvalid="skip")
    # df = assembler.transform(df)

    # scaler = MinMaxScaler(inputCol="numeric_features", outputCol="scaled_features")
    # df = scaler.fit(df).transform(df)
    
    tokenizer = Tokenizer(inputCol="review_body", outputCol="review_tokens")
    df = tokenizer.transform(df)

    return df

### 4) Target Engineering & Sampling (Kayla)

- Calculate the target variable: helpfulness_ratio (helpful_votes / total_votes) for rows where total_votes > 0
- Create other new features using Spark SQL functions (log of helpful vote, log of total votes, log of review length)
- Implement sampleBy() for Stratified Sampling to address the heavy skew toward 5-star ratings
- Perform the final Train/Validation/Test split (e.g., 80/10/10) for model evaluation.

In [ ]:
from pyspark.sql.functions import when, log1p

def cleansing4(df):
    # Target variable
    df = df.filter(col("total_votes") > 0)
    df = df.withColumn("helpfulness_ratio", col("helpful_votes") / col("total_votes"))
    df = df.withColumn("label",
        when(col("helpful_votes") / col("total_votes") >= 0.5, 1.0).otherwise(0.0)
    )
    # Feature engineering using Spark SQL functions
    df = df.withColumn("log_helpful_votes", log1p(col("helpful_votes")))
    df = df.withColumn("log_total_votes", log1p(col("total_votes")))
    df = df.withColumn("log_review_length", log1p(col("review_length")))

    # Stratified Sampling
    fractions = {1: 0.6, 2: 1.0, 3: 0.67, 4: 0.32, 5: 0.09}
    df = df.sampleBy("star_rating", fractions=fractions, seed=42)
    return df

### Pipelines

In [ ]:
# df_clean.write.mode("overwrite").parquet("/expanse/lustre/projects/uci157/hkwon2/shared/amazon_reviews_clean.parquet")

In [ ]:
#=== df_clean is stored in "/expanse/lustre/projects/uci157/hkwon2/shared/amazon_reviews_clean.parquet". No need to run this cell ===
#checkpoint_dir = "/expanse/lustre/projects/uci157/hkwon2/spark_checkpoints"
#spark.sparkContext.setCheckpointDir(checkpoint_dir)

#df_clean = cleansing4(cleansing3(cleansing2(cleansing1(df)))).checkpoint()

---
### Output of preprocessed clean dataset

In [9]:
output_path = "/expanse/lustre/projects/uci157/hkwon2/shared/amazon_reviews_clean.parquet"
df_clean = spark.read.parquet(output_path)

In [ ]:
df_count = df.count()
df_clean_count = df_clean.count()

print(f"Before filtering invalid records: {df_count}")
print(f"After clean up: {df_clean_count}")
print(f"{100*(df_count-df_clean_count)/df_count:.2f}% of noise data has been removed.")

In [ ]:
df_clean.select("label").groupBy("label").count().show()

Label 1 is 3 times larger than the other. When measuring accuracy, AUC and F1 scoring have to be used to evaluate considering the imbalance of data.

## 2. Train Your First Distributed Model (8 points)

### Feature Assembly & Train/Val/Test Split

In [10]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

feature_cols = [
    "star_rating",
    "review_length",
    "review_headline_length",
    # "review_year", (#seems irrelavent when predicting the future review)
    # "vine_idx", (#vine seems irrelavent to the helpfulness as stated above)
    "verified_purchase_idx",
    "category_idx"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

cols_needed = feature_cols + ["label"]

train_df, val_df, test_df = df_clean.select(cols_needed).randomSplit([0.7, 0.15, 0.15], seed=42)

In [11]:
train_df.show(5)

+-----------+-------------+----------------------+---------------------+------------+-----+
|star_rating|review_length|review_headline_length|verified_purchase_idx|category_idx|label|
+-----------+-------------+----------------------+---------------------+------------+-----+
|          1|            1|                     1|                  0.0|        24.0|  0.0|
|          1|            1|                     8|                  0.0|         0.0|  1.0|
|          1|            2|                     2|                  0.0|         3.0|  0.0|
|          1|            2|                     2|                  0.0|         5.0|  0.0|
|          1|            2|                     8|                  0.0|         3.0|  0.0|
+-----------+-------------+----------------------+---------------------+------------+-----+
only showing top 5 rows



In [12]:
evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

### Model 1: Random Forest (numTrees=20)

In [ ]:
rf20 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxBins=64, seed=42)
pipeline_rf20 = Pipeline(stages=[assembler, rf20])
model_rf20 = pipeline_rf20.fit(train_df)

train_pred_rf20 = model_rf20.transform(train_df)
val_pred_rf20   = model_rf20.transform(val_df)
test_pred_rf20  = model_rf20.transform(test_df)

print("=== Random Forest (numTrees=20) ===")
print(f"Train AUC: {evaluator_auc.evaluate(train_pred_rf20):.4f} | F1: {evaluator_f1.evaluate(train_pred_rf20):.4f}")
print(f"Val   AUC: {evaluator_auc.evaluate(val_pred_rf20):.4f} | F1: {evaluator_f1.evaluate(val_pred_rf20):.4f}")
print(f"Test  AUC: {evaluator_auc.evaluate(test_pred_rf20):.4f} | F1: {evaluator_f1.evaluate(test_pred_rf20):.4f}")

### Model 2: Random Forest (numTrees=50) — Hyperparameter Comparison

In [ ]:
rf50 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxBins=64, seed=42)
pipeline_rf50 = Pipeline(stages=[assembler, rf50])
model_rf50 = pipeline_rf50.fit(train_df)

train_pred_rf50 = model_rf50.transform(train_df)
val_pred_rf50   = model_rf50.transform(val_df)
test_pred_rf50  = model_rf50.transform(test_df)

print("=== Random Forest (numTrees=50) ===")
print(f"Train AUC: {evaluator_auc.evaluate(train_pred_rf50):.4f} | F1: {evaluator_f1.evaluate(train_pred_rf50):.4f}")
print(f"Val   AUC: {evaluator_auc.evaluate(val_pred_rf50):.4f} | F1: {evaluator_f1.evaluate(val_pred_rf50):.4f}")
print(f"Test  AUC: {evaluator_auc.evaluate(test_pred_rf50):.4f} | F1: {evaluator_f1.evaluate(test_pred_rf50):.4f}")

In [ ]:
rf50_d14 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=7, maxBins=64, seed=42)
pipeline_rf50_d14 = Pipeline(stages=[assembler, rf50_d14])
model_rf50_d14 = pipeline_rf50_d14.fit(train_df)

train_pred_rf50_d14 = model_rf50_d14.transform(train_df)
val_pred_rf50_d14   = model_rf50_d14.transform(val_df)
test_pred_rf50_d14  = model_rf50_d14.transform(test_df)


print("=== Random Forest (numTrees=50, maxDepth=7) ===")
print(f"Train AUC: {evaluator_auc.evaluate(train_pred_rf50_d14):.4f} | F1: {evaluator_f1.evaluate(train_pred_rf50_d14):.4f}")
print(f"Val   AUC: {evaluator_auc.evaluate(val_pred_rf50_d14):.4f} | F1: {evaluator_f1.evaluate(val_pred_rf50_d14):.4f}")
print(f"Test  AUC: {evaluator_auc.evaluate(test_pred_rf50_d14):.4f} | F1: {evaluator_f1.evaluate(test_pred_rf50_d14):.4f}")

### Model 3: GBT (maxIter=20, maxDepth=3)

In [13]:
from pyspark.ml.classification import GBTClassifier

In [ ]:


gbt_a20 = GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, maxDepth=3, maxBins=64, stepSize=0.1, seed=42)
pipeline_gbt_a20 = Pipeline(stages=[assembler, gbt_a20])

model_gbt_a_20 = pipeline_gbt_a20.fit(train_df)

train_pred_gbt_a20 = model_gbt_a_20.transform(train_df)
val_pred_gbt_a20   = model_gbt_a_20.transform(val_df)
test_pred_gbt_a20  = model_gbt_a_20.transform(test_df)

print("=== GBT Model A (maxIter=20, maxDepth=3) ===")
print(f"Train AUC: {evaluator_auc.evaluate(train_pred_gbt_a20):.4f} | F1: {evaluator_f1.evaluate(train_pred_gbt_a20):.4f}")
print(f"Val   AUC: {evaluator_auc.evaluate(val_pred_gbt_a20):.4f}  | F1: {evaluator_f1.evaluate(val_pred_gbt_a20):.4f}")
print(f"Test  AUC: {evaluator_auc.evaluate(test_pred_gbt_a20):.4f}  | F1: {evaluator_f1.evaluate(test_pred_gbt_a20):.4f}")

In [ ]:
gbt_a50 = GBTClassifier(featuresCol="features", labelCol="label", maxIter=50, maxDepth=3, maxBins=64, stepSize=0.1, seed=42)
pipeline_gbt_a50 = Pipeline(stages=[assembler, gbt_a50])

model_gbt_a50 = pipeline_gbt_a50.fit(train_df)

train_pred_gbt_a50 = model_gbt_a50.transform(train_df)
val_pred_gbt_a50   = model_gbt_a50.transform(val_df)
test_pred_gbt_a50  = model_gbt_a50.transform(test_df)

print("=== GBT Model A (maxIter=50, maxDepth=3) ===")
print(f"Train AUC: {evaluator_auc.evaluate(train_pred_gbt_a50):.4f} | F1: {evaluator_f1.evaluate(train_pred_gbt_a50):.4f}")
print(f"Val   AUC: {evaluator_auc.evaluate(val_pred_gbt_a50):.4f}  | F1: {evaluator_f1.evaluate(val_pred_gbt_a50):.4f}")
print(f"Test  AUC: {evaluator_auc.evaluate(test_pred_gbt_a50):.4f}  | F1: {evaluator_f1.evaluate(test_pred_gbt_a50):.4f}")

### Model Comparison

In [ ]:
results = []
for name, train_p, val_p, test_p in [
    ("RF (numTrees=20)", train_pred_rf20, val_pred_rf20, test_pred_rf20),
    ("RF (numTrees=50)", train_pred_rf50, val_pred_rf50, test_pred_rf50),
    ("GBT (maxIter=20, maxDepth=3)", train_pred_gbt_a20, val_pred_gbt_a20, test_pred_gbt_a20),
    ("GBT (maxIter=50, maxDepth=3)", train_pred_gbt_a50, val_pred_gbt_a50, test_pred_gbt_a50)
]:
    results.append({
        "Model":     name,
        "Train AUC": evaluator_auc.evaluate(train_p),
        "Test AUC":  evaluator_auc.evaluate(test_p),
        "Train F1":  evaluator_f1.evaluate(train_p),
        "Test F1":   evaluator_f1.evaluate(test_p),
        "Train ACC": evaluator_acc.evaluate(train_p),
        "Test ACC":  evaluator_acc.evaluate(test_p)
    })

comparison_df = pd.DataFrame(results).set_index("Model")
print(comparison_df.round(4).to_string())

### Example ground truth and predictions (numTree=20) for train, validation, and test sets

In [ ]:
for split_name, pred_df in [("Train", train_pred_rf20), ("Validation", val_pred_rf20), ("Test", test_pred_rf20)]:
    print(f"\n[{split_name}]")
    pred_df.select("label", "prediction", "star_rating", "review_length", "review_headline_length", "verified_purchase_idx", "category_idx").show(5, truncate=False)

### Feature Importance (GBT)

In [ ]:
importances = model_gbt_a.stages[-1].featureImportances.toArray()

feat_df = pd.DataFrame({
    "feature":    feature_cols,
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feat_df["feature"], feat_df["importance"], color="steelblue")
plt.gca().invert_yaxis()
plt.title("Feature Importances — Random Forest (numTrees=20)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print(feat_df.to_string(index=False))

## 5 Speedup Analysis

We measure the speedup of distributed training by running GBT training with **1 executor** vs. the **full allocation (15 executors)** on SDSC Expanse.

In [17]:
#Calculate for 1 executor
train_df.cache()
train_df.count() 

rf_speed = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=7, maxBins=64, seed=42)
pipeline_speed = Pipeline(stages=[assembler, rf_speed])

start = time.time()
pipeline_speed.fit(train_df.coalesce(1))
T_1 = time.time() - start

# ── T_n: full 15 executors ────────────────────────────────────────────────────
rf_speed2 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=7, maxBins=64, seed=42)
pipeline_speed2 = Pipeline(stages=[assembler, rf_speed2])

start = time.time()
pipeline_speed2.fit(train_df)
T_n = time.time() - start


In [18]:
n = 15
speedup    = T_1 / T_n
efficiency = speedup / n
p = (n * (speedup - 1)) / (speedup * (n - 1))  # parallelizable fraction
amdahl_max = 1 / (1 - p)

print(f"T_1 (coalesce=1, serial baseline): {T_1:.2f} s")
print(f"T_n (15 executors, distributed): {T_n:.2f} s")
print(f"\nSpeedup             : {speedup:.2f}x")
print(f"Efficiency          : {efficiency:.1%}")
print(f"Est. parallelizable : {p:.1%}")
print(f"Amdahl max speedup  : {amdahl_max:.2f}x")

# ── Summary table ─────────────────────────────────────────────────────────────
speedup_df = pd.DataFrame({
    "Executors" : [1, n],
    "Time (sec)": [f"{T_1:.2f}", f"{T_n:.2f}"],   # f-string instead of round()
    "Speedup"   : ["1.00x", f"{speedup:.2f}x"],
    "Efficiency": ["100.0%", f"{efficiency:.1%}"],
}).set_index("Executors")

print()
print(speedup_df.to_string())

T_1 (coalesce=1, serial baseline): 609.76 s
T_n (15 executors, distributed): 75.21 s

Speedup             : 8.11x
Efficiency          : 54.1%
Est. parallelizable : 93.9%
Amdahl max speedup  : 16.47x

          Time (sec) Speedup Efficiency
Executors                              
1             609.76   1.00x     100.0%
15             75.21   8.11x      54.1%
